# BART106M: обучение

## 1. Настройки и зависимости

In [ ]:
import os, sys, json, time, subprocess
from pathlib import Path
if 'torch' in sys.modules and sys.modules['torch'].cuda.is_initialized():
    raise RuntimeError('CUDA уже инициализирована')
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
PROJECT = Path.cwd()
if not (PROJECT / 'data.py').is_file():
    PROJECT = PROJECT / 'submission_final'
assert (PROJECT / 'data.py').is_file(), 'data.py не найден'
sys.path.insert(0, str(PROJECT))
ASSETS = PROJECT / 'data'
PREPARED = ASSETS / 'colab_training_data'
TRAIN_OUTPUT = PROJECT / 'weights/training'
NEW_AVERAGE = PROJECT / 'weights/avg3_mid_rebuilt'
RESUME = None
CONFIG = dict(seed=42, device="cuda", precision="bf16", physical_batch=128, effective_batch=256,
              peak_lr=5e-4, warmup_steps=3147, planned_steps=104908, floor_lr=5e-5,
              clip_grad=1.0, stop_step=102308,
              save_steps=[90940, 96625, 102308], save_every=2000)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT/'requirements.txt')], check=True)



## 2. Данные

In [ ]:
from data import unpack_prepared, verify_data
IDENTITY = json.loads((ASSETS / 'identity.json').read_text())
PREPARED = unpack_prepared(ASSETS)
TRAIN, VALIDATION = verify_data(PREPARED, ASSETS)
print('Train:', len(TRAIN), 'validation:', len(VALIDATION))


## 3. Модель

In [ ]:
from model import model_config
print(model_config())


## 4. Обучение

In [ ]:
from train import fit
STATE = fit(TRAIN, VALIDATION, CONFIG, IDENTITY, TRAIN_OUTPUT, resume=RESUME)


## 5. Усреднение

In [ ]:
from model import average_checkpoints
WEIGHTS = average_checkpoints(
    [TRAIN_OUTPUT / f'step{step:09d}' for step in CONFIG['save_steps']], NEW_AVERAGE)
print('Веса:', WEIGHTS)


## 6. Инференс

In [ ]:
inference_started = time.perf_counter()
from inference import run_inference
submission, timing = run_inference(
    WEIGHTS, ASSETS, PROJECT / 'outputs', started=inference_started, historical=False)
